<a href="https://colab.research.google.com/github/ZataraHere/Basic-Neural-Networks/blob/main/AutoEncoder_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# An **Autoencoder** is an unsupervised neural network tries to learn a function such that the output $\hat{x}$ is equal to the input $x$ ($x \to \hat{x}$).

# To train an autoencoder, we measure how poorly it reconstructs the input using a reconstruction loss, such as Mean Squared Error (MSE):$$L(x, \hat{x}) = \frac{1}{2} \vert{}\vert{}x - \hat{x}\vert{}\vert{}^2$$

In [2]:
import numpy as np

np.random.seed(42)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))  # Clipped for numerical stability

def sigmoid_derivative(sig):
    return sig * (1 - sig)

class SimpleAutoencoder:
    def __init__(self, input_dim, hidden_dim):


        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros((1, hidden_dim))


        self.W2 = np.random.randn(hidden_dim, input_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros((1, input_dim))

    def forward(self, X):

        self.X = X
        # Encoder pass
        self.z1 = np.dot(X, self.W1) + self.b1
        self.h = sigmoid(self.z1)

        # Decoder pass
        self.z2 = np.dot(self.h, self.W2) + self.b2
        self.x_hat = sigmoid(self.z2)

        return self.x_hat

    def backward(self, learning_rate):
        m = self.X.shape[0]  # Number of samples

        # Loss derivative w.r.t output: dL/d(x_hat) = (x_hat - X)
        error = self.x_hat - self.X

        # Gradients for Decoder
        dz2 = error * sigmoid_derivative(self.x_hat)
        dW2 = np.dot(self.h.T, dz2) / m
        db2 = np.sum(dz2, axis=0, keepdims=True) / m

        # Gradients for Encoder (backpropagate through the hidden layer)
        dh = np.dot(dz2, self.W2.T)
        dz1 = dh * sigmoid_derivative(self.h)
        dW1 = np.dot(self.X.T, dz1) / m
        db1 = np.sum(dz1, axis=0, keepdims=True) / m

        # Gradient Descent Update Step
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2

# ==========================================
# 3. Train on Synthetic Data
# ==========================================
if __name__ == "__main__":
    # Generate synthetic dataset: 1000 samples, 20 features (normalized between 0 and 1)
    # Let's make features have some redundancy so the autoencoder can compress them
    raw_data = np.random.rand(1000, 20)
    # Introduce dependency: feature 10 is a copy of feature 0, etc.
    raw_data[:, 10] = raw_data[:, 0]
    raw_data[:, 15] = raw_data[:, 5]

    input_dim = 20
    hidden_dim = 4  # Bottleneck layer (compressing 20 dims down to 4)

    ae = SimpleAutoencoder(input_dim=input_dim, hidden_dim=hidden_dim)

    epochs = 1000
    lr = 0.5

    print("Training Autoencoder from scratch...")
    for epoch in range(epochs):
        # Forward pass
        preds = ae.forward(raw_data)

        # Compute Mean Squared Error Loss
        loss = np.mean(np.square(preds - raw_data))

        # Backward pass & update weights
        ae.backward(learning_rate=lr)

        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch + 1}/{epochs} - Loss (MSE): {loss:.6f}")

    print("\nTraining complete!")
    print(f"Final Reconstruction Error: {loss:.6f}")

Training Autoencoder from scratch...
Epoch 200/1000 - Loss (MSE): 0.076480
Epoch 400/1000 - Loss (MSE): 0.070655
Epoch 600/1000 - Loss (MSE): 0.067126
Epoch 800/1000 - Loss (MSE): 0.064883
Epoch 1000/1000 - Loss (MSE): 0.063367

Training complete!
Final Reconstruction Error: 0.063367
